In [1]:
import cv2
import numpy as np
import time

from pynq import Overlay, MMIO, allocate
from pynq.lib.video import VideoMode
from pynq.lib import AxiGPIO

In [2]:
BITFILE = "AES_SYS.bit"   # 改成你的 bit 名稱，例如 "RFC8439.bit"
ol = Overlay(BITFILE, download=True)

print(ol.ip_dict.keys())

dict_keys(['axi_gpio_0', 'axi_gpio_1', 'axi_gpio_2', 'axi_gpio_3', 'axi_gpio_5', 'axi_gpio_6', 'axi_gpio_8', 'axi_gpio_10', 'axi_intc_0', 'axi_vdma_0', 'axi_cdma_0', 'processing_system7_0'])


In [3]:
# =========================================================
# Image / chunk config
# =========================================================
NUM_CHUNKS = 24
IMG_WIDTH  = 1280
IMG_HEIGHT = 720
BYTES_PER_PIXEL = 4

FULL_IMAGE_PIXELS = IMG_WIDTH * IMG_HEIGHT
FULL_IMAGE_BYTES  = FULL_IMAGE_PIXELS * BYTES_PER_PIXEL

assert FULL_IMAGE_PIXELS % NUM_CHUNKS == 0
PIXELS_PER_CHUNK = FULL_IMAGE_PIXELS // NUM_CHUNKS
CHUNK_BYTES = PIXELS_PER_CHUNK * BYTES_PER_PIXEL

TAG_SIZE = 16

# =========================================================
# RFC8439 layout
# Encrypt:
#   SrcRAM = [key][nonce][AAD][padding][plaintext]
#   DstRAM = [ciphertext][tag]
#
# Decrypt:
#   SrcRAM = [key][nonce][AAD][padding][ciphertext][tag]
#   DstRAM = [plaintext]
# =========================================================
KEY = bytes.fromhex(
    "000102030405060708090a0b0c0d0e0f"
    "101112131415161718191a1b1c1d1e1f"
)

BASE_NONCE = bytes.fromhex("000000090000004a00000000")

# 先用空 AAD，最簡單。若要放 metadata，可以改這裡。
AAD = b""

def align16(x):
    return (x + 15) & ~15

AD_LEN = len(AAD)
SRC_AAD_BASE_BYTES = 44   # key 32B + nonce 12B
MSG_OFFSET = align16(SRC_AAD_BASE_BYTES + AD_LEN)

SRC_ENC_BYTES = MSG_OFFSET + CHUNK_BYTES
SRC_DEC_BYTES = MSG_OFFSET + CHUNK_BYTES + TAG_SIZE
DST_ENC_BYTES = CHUNK_BYTES + TAG_SIZE
DST_DEC_BYTES = CHUNK_BYTES

print("CHUNK_BYTES   =", CHUNK_BYTES)
print("AD_LEN        =", AD_LEN)
print("MSG_OFFSET    =", MSG_OFFSET)
print("SRC_ENC_BYTES =", SRC_ENC_BYTES)
print("SRC_DEC_BYTES =", SRC_DEC_BYTES)

CHUNK_BYTES   = 153600
AD_LEN        = 0
MSG_OFFSET    = 48
SRC_ENC_BYTES = 153648
SRC_DEC_BYTES = 153664


In [4]:
# =========================================================
# IP base addresses
# =========================================================
GPIO1_ADDR = ol.ip_dict["axi_gpio_1"]["phys_addr"]   # msg_length
GPIO2_ADDR = ol.ip_dict["axi_gpio_2"]["phys_addr"]   # ad_length
GPIO3_ADDR = ol.ip_dict["axi_gpio_3"]["phys_addr"]   # mac_error
GPIO5_ADDR = ol.ip_dict["axi_gpio_5"]["phys_addr"]   # start
GPIO6_ADDR = ol.ip_dict["axi_gpio_6"]["phys_addr"]   # mode_decrypt
GPIO8_ADDR = ol.ip_dict["axi_gpio_8"]["phys_addr"]   # done

GPIO1_RANGE = ol.ip_dict["axi_gpio_1"]["addr_range"]
GPIO2_RANGE = ol.ip_dict["axi_gpio_2"]["addr_range"]
GPIO3_RANGE = ol.ip_dict["axi_gpio_3"]["addr_range"]
GPIO5_RANGE = ol.ip_dict["axi_gpio_5"]["addr_range"]
GPIO6_RANGE = ol.ip_dict["axi_gpio_6"]["addr_range"]
GPIO8_RANGE = ol.ip_dict["axi_gpio_8"]["addr_range"]

CDMA_ADDR  = ol.ip_dict["axi_cdma_0"]["phys_addr"]
CDMA_RANGE = ol.ip_dict["axi_cdma_0"]["addr_range"]

MSG_LENGTH = MMIO(GPIO1_ADDR, GPIO1_RANGE)
AD_LENGTH  = MMIO(GPIO2_ADDR, GPIO2_RANGE)
MAC_ERROR  = MMIO(GPIO3_ADDR, GPIO3_RANGE)
START      = MMIO(GPIO5_ADDR, GPIO5_RANGE)
MODE_DEC   = MMIO(GPIO6_ADDR, GPIO6_RANGE)
DONE       = MMIO(GPIO8_ADDR, GPIO8_RANGE)
cdma       = MMIO(CDMA_ADDR, CDMA_RANGE)

BRAM0_ADDR = 0xC0000000   # SrcRAM
BRAM1_ADDR = 0xC2000000   # DstRAM

In [5]:
def cdma_reset(timeout=1.0):
    cdma.write(0x00, 0x4)

    t0 = time.time()
    while (cdma.read(0x00) & 0x4) != 0:
        if time.time() - t0 > timeout:
            raise TimeoutError(f"CDMA reset timeout, status=0x{cdma.read(0x04):08x}")

    # Clear pending interrupt bits: IOC / delay / error
    cdma.write(0x04, 0x7000)


def cdma_transfer(src_addr, dst_addr, nbytes, timeout=5.0):
    cdma_reset()

    cdma.write(0x00, 0x1)          # RS = 1
    cdma.write(0x18, int(src_addr))
    cdma.write(0x20, int(dst_addr))
    cdma.write(0x28, int(nbytes))  # BTT last, transfer starts here

    t0 = time.time()

    while True:
        status = cdma.read(0x04)

        if status & 0x770:
            raise RuntimeError(f"CDMA error, status=0x{status:08x}")

        # IOC_Irq bit means transfer complete
        if status & 0x1000:
            break

        if time.time() - t0 > timeout:
            raise TimeoutError(f"CDMA timeout, status=0x{status:08x}")

    # Clear IOC after completion
    cdma.write(0x04, 0x1000)
def run_rfc8439(mode_decrypt, msg_len, ad_len, timeout=20.0):
    START.write(0x0, 0)
    time.sleep(0.001)

    MODE_DEC.write(0x0, 1 if mode_decrypt else 0)
    MSG_LENGTH.write(0x0, int(msg_len))
    AD_LENGTH.write(0x0, int(ad_len))

    time.sleep(0.001)

    START.write(0x0, 1)
    time.sleep(0.001)

    t0 = time.time()

    # If previous DONE is still high, keep START high until core accepts start and clears DONE.
    while (DONE.read(0x0) & 0x1) != 0:
        if time.time() - t0 > timeout:
            START.write(0x0, 0)
            raise TimeoutError(
                f"DONE did not clear after start, "
                f"DONE=0x{DONE.read(0x0):08x}, "
                f"MAC_ERROR=0x{MAC_ERROR.read(0x0):08x}"
            )

    START.write(0x0, 0)

    while (DONE.read(0x0) & 0x1) == 0:
        if time.time() - t0 > timeout:
            raise TimeoutError(
                f"RFC8439 timeout waiting done, "
                f"DONE=0x{DONE.read(0x0):08x}, "
                f"MAC_ERROR=0x{MAC_ERROR.read(0x0):08x}"
            )

    return MAC_ERROR.read(0x0) & 0x1

In [6]:
# =========================================================
# Nonce helper
# 每個 chunk 當成一筆獨立 AEAD message，所以 nonce 建議每塊不同。
# 這裡用 BASE_NONCE 的最後 32-bit 加 chunk index。
# =========================================================
def nonce_for_chunk(chunk_idx):
    n = bytearray(BASE_NONCE)
    tail = int.from_bytes(n[8:12], "little")
    n[8:12] = (tail + chunk_idx).to_bytes(4, "little")
    return bytes(n)

def fill_src_header(buf, nonce):
    buf[:] = 0
    buf[0:32] = np.frombuffer(KEY, dtype=np.uint8)
    buf[32:44] = np.frombuffer(nonce, dtype=np.uint8)

    if AD_LEN > 0:
        buf[44:44 + AD_LEN] = np.frombuffer(AAD, dtype=np.uint8)
def run_rfc8439(mode_decrypt, msg_len, ad_len, timeout=20.0):
    START.write(0x0, 0)
    time.sleep(0.001)

    MODE_DEC.write(0x0, 1 if mode_decrypt else 0)
    MSG_LENGTH.write(0x0, int(msg_len))
    AD_LENGTH.write(0x0, int(ad_len))

    time.sleep(0.001)

    START.write(0x0, 1)
    time.sleep(0.001)

    t0 = time.time()

    # If previous DONE is still high, keep START high until core accepts start and clears DONE.
    while (DONE.read(0x0) & 0x1) != 0:
        if time.time() - t0 > timeout:
            START.write(0x0, 0)
            raise TimeoutError(
                f"DONE did not clear after start, "
                f"DONE=0x{DONE.read(0x0):08x}, "
                f"MAC_ERROR=0x{MAC_ERROR.read(0x0):08x}"
            )

    START.write(0x0, 0)

    while (DONE.read(0x0) & 0x1) == 0:
        if time.time() - t0 > timeout:
            raise TimeoutError(
                f"RFC8439 timeout waiting done, "
                f"DONE=0x{DONE.read(0x0):08x}, "
                f"MAC_ERROR=0x{MAC_ERROR.read(0x0):08x}"
            )

    return MAC_ERROR.read(0x0) & 0x1

In [7]:
# =========================================================
# Load image and pack each pixel as 0x00RRGGBB
# =========================================================
INPUT_IMAGE = "input.jpg"   # 改成你的圖片檔名，不建議用 dec.png 當輸入

img_bgr = cv2.imread(INPUT_IMAGE)
if img_bgr is None:
    raise RuntimeError(f"Cannot read image: {INPUT_IMAGE}")

img_bgr = cv2.resize(img_bgr, (IMG_WIDTH, IMG_HEIGHT))
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

r = img_rgb[:, :, 0].astype(np.uint32)
g = img_rgb[:, :, 1].astype(np.uint32)
b = img_rgb[:, :, 2].astype(np.uint32)

plain_words = ((r << 16) | (g << 8) | b).astype(np.uint32).reshape(-1)
plain_words = np.ascontiguousarray(plain_words)
plain_bytes = plain_words.view(np.uint8)

print("plain_words:", plain_words.shape)
print("plain_bytes:", plain_bytes.shape)

plain_words: (921600,)
plain_bytes: (3686400,)


In [8]:
# =========================================================
# DMA buffers
# =========================================================
src_dma = allocate(shape=(SRC_DEC_BYTES,), dtype=np.uint8, cacheable=False)
dst_dma = allocate(shape=(DST_ENC_BYTES,), dtype=np.uint8, cacheable=False)

enc_bytes = np.zeros((FULL_IMAGE_BYTES,), dtype=np.uint8)
dec_bytes = np.zeros((FULL_IMAGE_BYTES,), dtype=np.uint8)
tags = np.zeros((NUM_CHUNKS, TAG_SIZE), dtype=np.uint8)

In [9]:
# =========================================================
# Encrypt image chunks
# =========================================================
for i in range(NUM_CHUNKS):
    print(f"Encrypt chunk {i+1}/{NUM_CHUNKS}")

    pt_start = i * CHUNK_BYTES
    pt_end   = pt_start + CHUNK_BYTES

    nonce = nonce_for_chunk(i)

    fill_src_header(src_dma, nonce)
    src_dma[MSG_OFFSET:MSG_OFFSET + CHUNK_BYTES] = plain_bytes[pt_start:pt_end]

    src_dma.flush()

    # SrcRAM = [key][nonce][AAD][padding][plaintext]
    cdma_transfer(src_dma.physical_address, BRAM0_ADDR, SRC_ENC_BYTES)

    # mode_decrypt = 0
    mac_err = run_rfc8439(
        mode_decrypt=False,
        msg_len=CHUNK_BYTES,
        ad_len=AD_LEN,
        timeout=20.0
    )

    if mac_err:
        print("Warning: mac_error asserted during encrypt")

    # DstRAM = [ciphertext][tag]
    cdma_transfer(BRAM1_ADDR, dst_dma.physical_address, DST_ENC_BYTES)
    dst_dma.invalidate()

    enc_bytes[pt_start:pt_end] = np.array(dst_dma[:CHUNK_BYTES], dtype=np.uint8)
    tags[i, :] = np.array(dst_dma[CHUNK_BYTES:CHUNK_BYTES + TAG_SIZE], dtype=np.uint8)

print("Encrypt done")

Encrypt chunk 1/24
Encrypt chunk 2/24
Encrypt chunk 3/24
Encrypt chunk 4/24
Encrypt chunk 5/24
Encrypt chunk 6/24
Encrypt chunk 7/24
Encrypt chunk 8/24
Encrypt chunk 9/24
Encrypt chunk 10/24
Encrypt chunk 11/24
Encrypt chunk 12/24
Encrypt chunk 13/24
Encrypt chunk 14/24
Encrypt chunk 15/24
Encrypt chunk 16/24
Encrypt chunk 17/24
Encrypt chunk 18/24
Encrypt chunk 19/24
Encrypt chunk 20/24
Encrypt chunk 21/24
Encrypt chunk 22/24
Encrypt chunk 23/24
Encrypt chunk 24/24
Encrypt done


In [10]:
# =========================================================
# Decrypt image chunks
# =========================================================
auth_fail = False

for i in range(NUM_CHUNKS):
    print(f"Decrypt chunk {i+1}/{NUM_CHUNKS}")

    ct_start = i * CHUNK_BYTES
    ct_end   = ct_start + CHUNK_BYTES

    nonce = nonce_for_chunk(i)

    fill_src_header(src_dma, nonce)
    src_dma[MSG_OFFSET:MSG_OFFSET + CHUNK_BYTES] = enc_bytes[ct_start:ct_end]
    src_dma[MSG_OFFSET + CHUNK_BYTES:MSG_OFFSET + CHUNK_BYTES + TAG_SIZE] = tags[i]

    src_dma.flush()

    # SrcRAM = [key][nonce][AAD][padding][ciphertext][tag]
    cdma_transfer(src_dma.physical_address, BRAM0_ADDR, SRC_DEC_BYTES)

    # mode_decrypt = 1
    mac_err = run_rfc8439(
        mode_decrypt=True,
        msg_len=CHUNK_BYTES,
        ad_len=AD_LEN,
        timeout=20.0
    )

    if mac_err:
        auth_fail = True
        print(f"AUTH FAIL at chunk {i}")

    # DstRAM = [plaintext]
    cdma_transfer(BRAM1_ADDR, dst_dma.physical_address, DST_DEC_BYTES)
    dst_dma.invalidate()

    dec_bytes[ct_start:ct_end] = np.array(dst_dma[:CHUNK_BYTES], dtype=np.uint8)
print("Test XOR decrypt using encrypt mode on chunk 0")

i = 0
ct_start = i * CHUNK_BYTES
ct_end   = ct_start + CHUNK_BYTES

nonce = nonce_for_chunk(i)

fill_src_header(src_dma, nonce)
src_dma[MSG_OFFSET:MSG_OFFSET + CHUNK_BYTES] = enc_bytes[ct_start:ct_end]
src_dma.flush()

cdma_transfer(src_dma.physical_address, BRAM0_ADDR, SRC_ENC_BYTES)

# 故意用 encrypt mode
mac_err = run_rfc8439(
    mode_decrypt=False,
    msg_len=CHUNK_BYTES,
    ad_len=AD_LEN,
    timeout=20.0
)

cdma_transfer(BRAM1_ADDR, dst_dma.physical_address, DST_ENC_BYTES)
dst_dma.invalidate()

xor_dec0 = np.array(dst_dma[:CHUNK_BYTES], dtype=np.uint8)

print("Decrypt done")
print("AUTH:", "FAIL" if auth_fail else "PASS")
print("Round-trip:", "PASS" if np.array_equal(dec_bytes, plain_bytes) else "FAIL")


Decrypt chunk 1/24
Decrypt chunk 2/24
Decrypt chunk 3/24
Decrypt chunk 4/24
Decrypt chunk 5/24
Decrypt chunk 6/24
Decrypt chunk 7/24
Decrypt chunk 8/24
Decrypt chunk 9/24
Decrypt chunk 10/24
Decrypt chunk 11/24
Decrypt chunk 12/24
Decrypt chunk 13/24
Decrypt chunk 14/24
Decrypt chunk 15/24
Decrypt chunk 16/24
Decrypt chunk 17/24
Decrypt chunk 18/24
Decrypt chunk 19/24
Decrypt chunk 20/24
Decrypt chunk 21/24
Decrypt chunk 22/24
Decrypt chunk 23/24
Decrypt chunk 24/24
Test XOR decrypt using encrypt mode on chunk 0
Decrypt done
AUTH: PASS
Round-trip: PASS


In [11]:
# =========================================================
# Convert byte stream back to images
# =========================================================
def bytes_to_bgr_image(data_bytes):
    words = np.frombuffer(np.ascontiguousarray(data_bytes).tobytes(), dtype=np.uint32)
    words = words.reshape((IMG_HEIGHT, IMG_WIDTH))

    rr = ((words >> 16) & 0xFF).astype(np.uint8)
    gg = ((words >> 8)  & 0xFF).astype(np.uint8)
    bb = ( words        & 0xFF).astype(np.uint8)

    return cv2.merge([bb, gg, rr])

enc_img_bgr = bytes_to_bgr_image(enc_bytes)
dec_img_bgr = bytes_to_bgr_image(dec_bytes)

cv2.imwrite("rfc8439_enc.png", enc_img_bgr)
cv2.imwrite("rfc8439_dec.png", dec_img_bgr)

print("Saved rfc8439_enc.png")
print("Saved rfc8439_dec.png")

Saved rfc8439_enc.png
Saved rfc8439_dec.png


In [ ]:
# =========================================================
# Stable HDMI / VDMA display
# sw = 0: original
# sw = 1: encrypted
# sw >=2: decrypted
# =========================================================
import time
import numpy as np
import cv2
from pynq import allocate
from pynq.lib.video import VideoMode
from pynq.lib import AxiGPIO

# ---------------------------------------------------------
# HDMI display config
# ---------------------------------------------------------
IMG_WIDTH  = 1280
IMG_HEIGHT = 720

# 水平修正：以 pixel 為單位
# 你目前接近可用的是 -473
HDMI_X_SHIFT_PIXELS = -470

# RGB888 byte phase 修正：只能是 0, 1, 2
# 如果畫面顏色還是不對，可以只改這個值
HDMI_BYTE_PHASE = 1

# 顯示端顏色來源
# OpenCV 讀進來是 BGR；通常 HDMI/RGB2DVI 端吃 RGB，所以先轉 RGB
# 如果你的 HDMI 端吃 BGR，把這個改成 False
USE_BGR2RGB = True

# ---------------------------------------------------------
# Start VDMA
# ---------------------------------------------------------
vdma = ol.axi_vdma_0

try:
    vdma.writechannel.stop()
    time.sleep(0.1)
except Exception:
    pass

mode = VideoMode(IMG_WIDTH, IMG_HEIGHT, 24)
vdma.writechannel.mode = mode
vdma.writechannel.start()
time.sleep(0.1)

# ---------------------------------------------------------
# Switch GPIO
# ---------------------------------------------------------
sw_1 = AxiGPIO(ol.ip_dict["axi_gpio_10"]).channel1

# ---------------------------------------------------------
# Allocate HDMI frames
# ---------------------------------------------------------
original_frame = allocate(
    shape=(IMG_HEIGHT, IMG_WIDTH, 3),
    dtype=np.uint8,
    cacheable=False
)

enc_frame = allocate(
    shape=(IMG_HEIGHT, IMG_WIDTH, 3),
    dtype=np.uint8,
    cacheable=False
)

dec_frame = allocate(
    shape=(IMG_HEIGHT, IMG_WIDTH, 3),
    dtype=np.uint8,
    cacheable=False
)

# ---------------------------------------------------------
# HDMI image fix
# 1. 可選 BGR -> RGB
# 2. 用 byte-level shift 修正 RGB888 byte phase / horizontal offset
# ---------------------------------------------------------
def prepare_for_hdmi(img_bgr):
    if USE_BGR2RGB:
        frame = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    else:
        frame = img_bgr

    frame = np.ascontiguousarray(frame)

    # 以每一列為單位做 byte roll，避免整張圖跨列互相捲動
    row_bytes = IMG_WIDTH * 3
    byte_shift = HDMI_X_SHIFT_PIXELS * 3 + HDMI_BYTE_PHASE

    flat_rows = frame.reshape(IMG_HEIGHT, row_bytes)
    fixed_rows = np.roll(flat_rows, byte_shift, axis=1)

    return np.ascontiguousarray(fixed_rows.reshape(IMG_HEIGHT, IMG_WIDTH, 3))


# ---------------------------------------------------------
# Copy images into HDMI buffers
# ---------------------------------------------------------
original_frame[:] = prepare_for_hdmi(img_bgr)
enc_frame[:]      = prepare_for_hdmi(enc_img_bgr)
dec_frame[:]      = prepare_for_hdmi(dec_img_bgr)

original_frame.flush()
enc_frame.flush()
dec_frame.flush()

# ---------------------------------------------------------
# Frame list for switch control
# ---------------------------------------------------------
frames = [
    original_frame,
    enc_frame,
    dec_frame,
    dec_frame
]

names = [
    "original",
    "encrypted",
    "decrypted",
    "decrypted"
]

# ---------------------------------------------------------
# Initial frame
# ---------------------------------------------------------
vdma.writechannel.writeframe(original_frame)

print("HDMI display started")
print("frame shape:", original_frame.shape)
print("frame strides:", original_frame.strides)
print("original phys:", hex(original_frame.physical_address))
print("enc phys     :", hex(enc_frame.physical_address))
print("dec phys     :", hex(dec_frame.physical_address))
print("HDMI_X_SHIFT_PIXELS:", HDMI_X_SHIFT_PIXELS)
print("HDMI_BYTE_PHASE:", HDMI_BYTE_PHASE)
print("USE_BGR2RGB:", USE_BGR2RGB)

# ---------------------------------------------------------
# Switch loop
# ---------------------------------------------------------
last_val = -1

while True:
    val = sw_1.read() & 0x3

    if val != last_val:
        vdma.writechannel.writeframe(frames[val])
        print("show:", names[val])
        last_val = val

    time.sleep(0.05)